## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Python-dotenv could not parse statement starting at line 5


Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

/tmp/ipykernel_22807/4072587781.py:6: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name='fast')


2026-05-29 10:44:46,311 - INFO - root - PlanExecutor initialized with topology: fast
2026-05-29 10:44:46,311 - INFO - root -   Players per step: 2
2026-05-29 10:44:46,312 - INFO - root -   Debate rounds: 1
2026-05-29 10:44:46,312 - INFO - root -   Player pool: ['data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-05-29 10:44:46,312 - INFO - root - Orchestrator initialized with topology: fast
2026-05-29 10:44:46,312 - INFO - root - ============================================================
2026-05-29 10:44:46,313 - INFO - root - GENERATING PLAN
2026-05-29 10:44:46,313 - INFO - root - Context: biota_multi
2026-05-29 10:44:46,313 - INFO - root - Context type: multi_csv
2026-05-29 10:44:46,313 - INFO - root - Classified planning type: multi_csv
2026-05-29 10:44:46,313 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-05-29 10:44:46,314 - INFO - root - Is multi-context: True
2026-05-29 10:44:46,314 - IN

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:1150: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-05-29 10:44:46,651 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-05-29 10:44:46,653 - INFO - openai._base_client - Retrying request to /chat/completions in 0.473368 seconds
2026-05-29 10:44:47,352 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-05-29 10:44:47,353 - INFO - openai._base_client - Retrying request to /chat/completions in 0.940420 seconds
2026-05-29 10:44:48,478 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-05-29 10:44:48,480 - ERROR - root - Plan generation failed: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<title>503 Service Unavailable</title>
</head><body>
<h1>Service Unavailable</h1>
<p>The server is temporarily unable to service your
request due to maintenance downtime or capaci

In [ ]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

In [ ]:
plan.pretty_print()

In [ ]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

In [ ]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

In [ ]:
result.final_workspace